> 📄 English: [hybrid-agent-tutorial.ipynb](hybrid-agent-tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--hybrid-agent-tutorial)

## 3. 将内部数据与 Web 数据相结合 🌐🤝📊

欢迎来到教程 3！


在本教程中，你将学习如何构建一个**混合 Agent（hybrid agent）**，将实时 Web 数据与你自己的内部知识来源相结合。对企业而言，Agent 的真正威力来自将专有的内部数据与公开网络信息整合起来。

学完本教程，你将能够：

- **整合** Web 搜索结果与私有文档到单个 Agent 工作流中
- **让**你的 Agent 能够比较、对比并综合来自公开与内部来源的信息
- **构建**更准确、更新颖、更贴合上下文的应用程序，用于研究、分析和决策


> **为什么这种方法很有价值？**
> 它让你的 AI 能够用网络上最新的信息来回答问题，同时利用你的专有数据——就像人类专家会做的那样。
>
> 你将为市场研究、竞争情报，以及任何内部与外部知识都重要的场景解锁新能力。

---


## 快速开始

请按照以下步骤配置：

1. **注册** Tavily：访问 [app.tavily.com](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-with-tavily-web-access--hybrid-agent-tutorial&click=tavily-app&target=https%3A%2F%2Fapp.tavily.com%2Fhome%2F%3Futm_source%3Dgithub%26utm_medium%3Dreferral%26utm_campaign%3Dnir_diamant&text=app.tavily.com) 获取你的 API key。

   *可参考下方链接的截图，获取分步指引：*

   - [截图：注册页面](assets/sign-up.png)
   - [截图：Tavily API Keys 控制台](assets/api-key.png)


2. **注册** OpenAI 以获取你的 API key。默认情况下我们将使用 OpenAI——但你也可以替换为任何其他 LLM 提供商。


2. 从你的 Tavily 和 OpenAI 账号控制台**复制你的 API key**。

3. 将你的 API key**粘贴**到下面的单元格中并执行该单元格。


In [ ]:
# To export your API keys into a .env file, run the following cell (replace with your actual keys):
!echo "TAVILY_API_KEY=<your-tavily-api-key>" >> .env
!echo "OPENAI_API_KEY=<your-openai-api-key>" >> .env

在下面的单元格中安装依赖。

In [ ]:
%pip install -U tavily-python langchain-chroma langchain-openai langgraph langchain-tavily --quiet

### 配置你的 Tavily API 客户端

下面的代码将使用你的 API key 实例化 Tavily 客户端。

In [ ]:
import os
import getpass
from dotenv import load_dotenv
from tavily import TavilyClient

# Load environment variables from .env file
load_dotenv()

# Prompt the user to securely input the API key if not already set in the environment
if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:\n")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

# Initialize the Tavily API client using the loaded or provided API key
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

## 向量数据库配置

在本教程中，我们将使用一个预构建的向量数据库来模拟内部知识库。数据基于合成生成的模拟 CRM 文档，旨在支持一个 CRM 与销售洞察用例。

为简单起见，我们使用 Chroma 和 OpenAI embeddings 从模拟 CRM 数据构建了一个向量存储。你可以在 [supplemental/docs](supplemental/docs) 目录中自由探索底层文档。

要开始，只需运行下面的 Python 单元格来初始化向量索引。


In [ ]:
# Initialize the vector store
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Initialize the embeddings model with the default model
embeddings = OpenAIEmbeddings()

# Initialize the vector store
vector_store = Chroma(
    collection_name="crm",  # replace with "my_custom_index" for custom documents option
    embedding_function=embeddings,
    persist_directory="supplemental/db",
)

不过，如果你更愿意集成自己的文档，我们通过三个步骤让文件向量化变得简单：
1. 将你的文件（必须是 PDF）上传到 [supplemental/docs](supplemental/docs) 目录。

2. 按照 [`vectorize_tutorial.ipynb`](supplemental/vectorize_tutorial.ipynb) notebook 的步骤创建你的自定义向量索引。

3. 在上方单元格的 Chroma 实例化代码中，将 `collection_name="crm"` 替换为 `collection_name="my_custom_index"` 并重新运行。

注意：我们创建的向量数据库流水线仅用于演示，并未优化——鼓励你接入自己优化过的向量流水线、向量存储和 embeddings 模型，以适配你的数据与需求。 

让我们用一个简单的查询来测试向量索引。

In [ ]:
# Initialize the retriever
retriever = vector_store.as_retriever()

# Query the vector index
search_results = retriever.invoke("robotics use case")

让我们查看向量搜索的结果。

In [ ]:
# Print the results
for result in search_results:
    print(result.page_content)
    print("\n")

接下来，我们将配置工具：既包括教程 2 中引入的 Web 工具，也包括我们新的向量搜索工具。

In [ ]:
# Define the set of tools our agent will use
from langchain_tavily import TavilySearch
from langchain_tavily import TavilyExtract
from langchain_tavily import TavilyCrawl

# Define the vector search tool
vector_search_tool = retriever.as_tool(
    name="vector_search",
    description="""
    Perform a vector search on our company's CRM data.
    """,
)

# Define the LangChain search tool
search = TavilySearch(max_results=10, topic="general")

# Define the LangChain extract tool
extract = TavilyExtract(extract_depth="advanced")

# Define the LangChain crawl tool
crawl = TavilyCrawl()

我们将配置多个 OpenAI 基础模型，例如 o3-mini 和 gpt-4.1 模型。如果你更喜欢其他 LLM 提供商，可以轻松接入任何 LangChain Chat 模型。

In [ ]:
# Instantiate the OpenAI foundation models
from langchain_openai import ChatOpenAI

# o3-mini-2025-01-31
o3_mini = ChatOpenAI(model="o3-mini-2025-01-31", api_key=os.getenv("OPENAI_API_KEY"))

# gpt-4.1
gpt_4_1 = ChatOpenAI(model="gpt-4.1", api_key=os.getenv("OPENAI_API_KEY"))

## 混合 Agent

现在，我们将使用 LangGraph 定义混合 ReAct Agent，如下图所示。

<img src="./assets/hybrid.svg" alt="Agent" width="500"/>

系统提示词是引导 Agent 行为的关键组件，因为它为 Agent 如何与可用工具和数据源交互设定了上下文、边界和指令。一个精心设计的系统提示词能够确保 Agent 理解它可以访问哪些信息、如何有效使用每个工具，以及在生成回复时遵循哪些标准。

下面的系统提示词明确概述了 Agent 的能力，包括访问公开 Web 工具（search、extract、crawl）以及用于专有 CRM 数据的内部向量搜索工具。它详细说明了每个工具的目的和最佳实践，强调将回答建立在检索到的信息之上，并禁止 Agent 编造事实或依赖先验知识。这个全面的提示词帮助 Agent 就何时以及如何使用每个工具做出明智决策，从而产生更准确、更可靠的输出。

该提示词清楚描述了向量存储中有哪些信息可用，因此 Agent 知道如何将它
与 Web 工具一起使用。你可以自由试验系统提示词，并尝试不同的语言模型来调整
Agent 的行为，为你的需求优化结果。

In [ ]:
# Import the necessary libraries
import datetime
from langgraph.prebuilt import create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

today = datetime.datetime.today().strftime("%A, %B %d, %Y")

# Create the hybrid agent
hybrid_agent = create_react_agent(
    model=gpt_4_1,
    tools=[search, crawl, extract, vector_search_tool],
    prompt=ChatPromptTemplate.from_messages(
        [
            (
                "system",
                f"""
        You are a ReAct-style research agent equipped with the following tools:

        * **Tavily Web Search**
        * **Tavily Web Extract**
        * **Tavily Web Crawl**
        * **Internal Vector Search** (for proprietary CRM data)

        Your mission is to conduct comprehensive, accurate, and up-to-date research, using a combination of real-time public web information and internal enterprise knowledge.  All answers must be grounded in retrieved information from the tools provided. You are not permitted to make up information or rely on prior knowledge.

        **Today's Date:** {today}

        **Available Tools:**

        1. **Tavily Web Search**

        * **Purpose:** Retrieve relevant web pages based on a query.
        * **Usage:** Provide a search query to receive up to 10 semantically ranked results, each containing the title, URL, and a content snippet.
        * **Best Practices:**
            * Use specific queries to narrow down results.
                * Example: "OpenAI's latest product release" and "OpenAI's headquarters location" rather than "OpenAI's latest product release and headquarters location"
            * Optimize searches using parameters such as `search_depth`, `time_range`, `include_domains`, and `include_raw_content`.
            * Break down complex queries into specific, focused sub-queries.

        2. **Tavily Web Crawl**

        * **Purpose:** Explore a website's structure and gather content from linked pages for deep research and information discovery from a single source.
        * **Usage:** Input a base URL to crawl, specifying parameters such as `max_depth`, `max_breadth`, and `extract_depth`.
        * **Best Practices:**

            * Begin with shallow crawls and progressively increase depth.
            * Utilize `select_paths` or `exclude_paths` to focus the crawl.
            * Set `extract_depth` to "advanced" for comprehensive extraction.

        3. **Tavily Web Extract**

        * **Purpose:** Extract the full content from specific web pages.
        * **Usage:** Provide URLs to retrieve detailed content.
        * **Best Practices:**

            * Set `extract_depth` to "advanced" for detailed content, including tables and embedded media.
            * Enable `include_images` if image data is necessary.

        4. **Internal Vector Search**

        * **Purpose:** Search the proprietary CRM knowledge base.
        * **Usage:** Submit a natural language query to retrieve relevant context from the CRM vector store. Contains context about key accounts like Meta, Apple, Google, Amazon, Microsoft, and Tesla.
        * **Best Practices:**
            * When possible, refer to specific information, such as names, dates, product usage, or meetings.
            

        **Guidelines for Conducting Research:**

        * You may not answer questions based on prior knowledge or assumptions.
        * **Citations:** Always support findings with source URLs, clearly provided as in-text citations.
        * **Accuracy:** Rely solely on data obtained via provided tools (web or vector store) —never fabricate information.
        * **If none of the tools return useful information, respond:**
            * "I'm sorry, but none of the available tools provided sufficient information to answer this question."

        **Research Workflow:**

        * **Thought:** Consider necessary information and next steps.
        * **Action:** Select and execute appropriate tools.
        * **Observation:** Analyze obtained results.
        **IMPORTANT: Repeat Thought/Action/Observation cycles as needed. You MUST only respond to the user once you have gathered all the information you need.**
        * **Final Answer:** Synthesize and present findings with citations in markdown format.
        
        **Example Workflow:**

        **Workflow 4: Combine Web Tools and Vector Search**

        **Question:** What has Apple publicly shared about their AI strategy, and do we have any internal notes on recent meetings with them?

        * **Thought:** I’ll start by looking for Apple’s public statements on AI using a search.
        * **Action:** Tavily Web Search with the query "Apple AI strategy 2024" and `time_range` set to "month".
        * **Observation:** Found a recent article from Apple’s newsroom and a keynote transcript.
        * **Thought:** I want to read the full content of Apple’s keynote.
        * **Action:** Tavily Web Extract on the URL from the search result.
        * **Observation:** Extracted detailed content from Apple’s announcement.
        * **Thought:** Now, I’ll check our internal CRM data for recent meeting notes with Apple.
        * **Action:** Internal Vector Search with the query "recent Apple meetings AI strategy".
        * **Observation:** Retrieved notes from a Q2 strategy sync with Apple’s enterprise team.
        * **Final Answer:** Synthesized response combining public and internal data, with citations.
        ---

        You will now receive a research question from the user:
        """,
            ),
            MessagesPlaceholder(variable_name="messages"),
        ]
    ),
    name="hybrid_agent",
)

In [ ]:
from langchain_core.messages import HumanMessage

# Test the hybrid agent
inputs = {
    "messages": [
        HumanMessage(
            content="Search for the latest news on google relevant to our current CRM data on them"
        )
    ]
}

# Stream the agent's response
for s in hybrid_agent.stream(inputs, stream_mode="values"):
    message = s["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()

查看上方 Agent 的中间轨迹，了解 Agent 是如何将向量搜索和 Web 搜索结合在一起的。

让我们查看最终输出。 

In [ ]:
# Print the final output as markdown
from IPython.display import Markdown

Markdown(message.content)

如图所示，最终报告将相关的 Web 洞察与我们现有的 Google Cloud ML Ops CRM 数据整合在一起。

让我们再试一个例子。我们将询问 Agent："Check for the Google deal size and find the latest earnings report for them to determine if they are currently on a spending spree."

In [ ]:
# Test the hybrid agent
inputs = {
    "messages": [
        HumanMessage(
            content="Check for the Google Deal size and find the latest earnings report for them to validate if they are in spending spree"
        )
    ]
}

# Stream the agent's response
for s in hybrid_agent.stream(inputs, stream_mode="values"):
    message = s["messages"][-1]
    if isinstance(message, tuple):
        print(message)
    else:
        message.pretty_print()

让我们查看最终输出。

In [ ]:
# Print the final output as markdown
Markdown(message.content)

# 结论

恭喜你完成本教程——以及整个系列——关于使用 Tavily API 构建具有 Web 访问能力的智能 Agent！

通过这些教程，你已经学会了如何：

- **配置 Tavily API** 以实现实时 Web 搜索、提取和抓取。
- **设计 Agent 架构**，将 Web 与内部知识来源相结合。
- **集成多个工具**——包括 Web 搜索、提取、抓取和向量搜索——到一个统一而强大的 Agent 系统中。
- **将这些 Agent 应用**到现实场景中，例如研究公司和分析新闻。

借助 Tavily 的 Web API，你的 Agent 现在可以访问超出其训练数据的最新信息，使它们在动态业务中更加高效和贴合实际。

祝贺你完成了整个系列！我们希望这些技能能帮助你构建更先进、更具洞察力的 AI 解决方案。祝你编码愉快！
